# 03: Reversible transformer at the maximum batch size
Search for the largest batch the winning variant can run, back off 10% for headroom, and train on the same 50M tokens. The learning rate is scaled by √(B/B_base) and capped at 3e-3.

## Setup
Runtime → Change runtime type → **T4 GPU**. Results are written to Google Drive, so a disconnect does not lose finished runs.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/reversible-llm'
import os
os.makedirs(f'{DRIVE}/data', exist_ok=True)
os.makedirs(f'{DRIVE}/results', exist_ok=True)

In [ ]:
import os, sys
if not os.path.exists('/content/llm_finetune_assignments'):
    !git clone -q https://github.com/AditeyaAItronics/llm_finetune_assignments.git /content/llm_finetune_assignments
%cd /content/llm_finetune_assignments/reversible-llm-13
!git pull -q
!pip install -q tiktoken datasets
sys.path.insert(0, '/content/llm_finetune_assignments/reversible-llm-13/src')

In [ ]:
from revllm.data import prepare_tinystories
prepare_tinystories(f'{DRIVE}/data')  # tokenizes once (~55M tokens), cached on Drive
!mkdir -p /content/data && cp -n {DRIVE}/data/*.bin /content/data/

In [ ]:
import pytest
# includes the GPU memory test that is skipped on CPU
assert pytest.main(['-q', 'tests']) == 0, 'tests failed: fix before training'

In [ ]:
import json, math
from revllm.config import ModelConfig, TrainConfig
from revllm.batch_search import find_max_batch
from revllm.train import train

w = json.load(open(f'{DRIVE}/winner.json'))
B_base = json.load(open(f'{DRIVE}/results/01_baseline.json'))['batch_size']
mc = ModelConfig(arch=w['arch'], h=w['h'])
search = find_max_batch(mc, start=B_base)
json.dump(search, open(f"{DRIVE}/batch_search_{w['arch']}.json", 'w'))
search

In [ ]:
B_max = search['recommended']
lr = min(3e-3, 1e-3 * math.sqrt(B_max / B_base))
res = train(mc, TrainConfig(run_name=f"03_reversible_{w['arch']}_max_batch", batch_size=B_max,
                            lr=lr, min_lr=lr / 10,
                            data_dir='/content/data', out_dir=f'{DRIVE}/results'))
{k: res[k] for k in ['batch_size', 'steps', 'tokens_seen', 'final_train_loss', 'final_val_loss',
                     'tokens_per_sec', 'peak_mem_allocated_gb', 'wall_time_s']}

**Isolating the memory saving.** `find_max_batch` above uses the memory-saving reversible backward. Re-running it with `use_rev_fn=False` (plain autograd, no activation reconstruction) at the same starting batch isolates how much of the memory saving comes from the custom backward's activation reconstruction, versus the two-stream reversible architecture itself.

In [ ]:
search_no_revfn = find_max_batch(mc, start=B_base, use_rev_fn=False)
json.dump(search_no_revfn, open(f"{DRIVE}/batch_search_{w['arch']}_no_revfn.json", 'w'))
search_no_revfn